# Backend Tutorial: Simulate, Profile & FPGA

CPU simulation, energy profiling with device presets, and FPGA compilation.

In [ ]:
import numpy as np
from talon import ir, backend

## 1. Backends

In [ ]:
print(f"Backends: {backend.list_backends()}")
cpu = backend.get_backend("cpu")
print(f"CPU precisions: {cpu.get_capabilities().supported_precisions}")

In [ ]:
for name, p in backend.ENERGY_PRESETS.items():
    print(f"  {name}: {p['mac_pj']} pJ/MAC")

## 2. Graph

In [ ]:
nodes = {
    "input": ir.Input(np.array([784])),
    "fc1": ir.Affine(weight=np.random.randn(128, 784).astype(np.float32)*0.01, bias=np.zeros(128, dtype=np.float32)),
    "lif1": ir.LIF(tau=np.ones(128, dtype=np.float32)*10, r=np.ones(128, dtype=np.float32), v_leak=np.zeros(128, dtype=np.float32), v_threshold=np.ones(128, dtype=np.float32)),
    "fc2": ir.Affine(weight=np.random.randn(10, 128).astype(np.float32)*0.01, bias=np.zeros(10, dtype=np.float32)),
    "lif2": ir.LIF(tau=np.ones(10, dtype=np.float32)*10, r=np.ones(10, dtype=np.float32), v_leak=np.zeros(10, dtype=np.float32), v_threshold=np.ones(10, dtype=np.float32)),
    "output": ir.Output(np.array([10])),
}
edges = [("input","fc1"),("fc1","lif1"),("lif1","fc2"),("fc2","lif2"),("lif2","output")]
graph = ir.Graph(nodes=nodes, edges=edges)

## 3. Validate & Compile

In [ ]:
val = cpu.validate(graph)
print(f"Valid: {val.is_valid}")
compiled = cpu.compile(graph)
print(f"Compiled: {type(compiled).__name__}")

## 4. Simulate

In [ ]:
x = np.random.randn(1, 784).astype(np.float32)
sim = cpu.simulate(graph, {"input": x}, n_steps=3)
print(f"Steps: {sim.timesteps_run}, Valid: {sim.outputs_valid}")
print(f"Spikes: {sim.spike_counts}")

## 5. Profile

In [ ]:
prof = cpu.profile(graph)
print(f"Latency: {prof.total_latency_us:.2f} us")
print(f"MACs: {prof.mac_ops:,}")
print(f"Energy: {prof.energy_estimate_uj:.4f} uJ ({prof.energy_preset})")
print(f"Memory: {prof.peak_memory_bytes:,} bytes")
for layer, e in prof.per_layer_energy_uj.items():
    print(f"  {layer}: {e:.4f} uJ")

In [ ]:
prof_zcu = cpu.profile(graph, energy_preset="zcu102")
print(f"ZCU102: {prof_zcu.energy_estimate_uj:.4f} uJ")

## 6. FPGA

In [ ]:
fpga = backend.get_backend("fpga")
print(f"FPGA valid: {fpga.validate(graph).is_valid}")
print(f"Precisions: {fpga.get_capabilities().supported_precisions}")